# 05 - Critical Station Isolation

High betweenness centrality tells us that a stop carries many shortest paths through the
network - but it does **not** tell us that losing the stop would actually strand anyone.
If another stop sits 80 metres down the street, a passenger simply walks there and the
neighbourhood stays connected. The genuinely fragile points are the stops that are both
*structurally* critical **and** *spatially* isolated: no alternative within walking distance.

This notebook is the "reality check" on the centrality results. It takes the per-stop metrics
produced by the centrality stage, defines **critical = top 10% of betweenness**, and then uses
a haversine `BallTree` to measure, for every stop, the great-circle distance to its nearest
neighbouring stop. Critical stops whose nearest alternative is farther than **300 m** are
flagged as isolated single points of failure.

**Research question:** *Of the stops the graph calls critical, how many are critical in
practice - i.e. have no walkable substitute?*

### Inputs
- `outputs/nb/04_centrality_analysis/tables/stop_metrics.csv` - one row per graph node with
  `stop_id`, `stop_name`, latitude, longitude and betweenness centrality.
  (Produced by notebook `04` - this notebook does not recompute centrality.)

### Outputs (all under `outputs/nb/05_critical_station_isolation/`)
- `tables/critical_isolation.csv` - every critical stop, its distance to the nearest
  alternative stop, and an `is_isolated` flag.
- `tables/isolation_summary.csv` - the distance-band counts behind the main figure.
- `figures/critical_isolation.png` - **the presentation figure**: critical stops binned by
  distance to their nearest alternative, coloured green (substitutable) vs red (isolated).
- `figures/top_isolated_critical_stops.png` - the individual isolated stops with the highest
  betweenness, named.

### Important limitation (stated up front, repeated at the end)
"Has a stop within 300 m" is a purely **spatial** test. It ignores whether that neighbouring
stop is served by the same lines, in the same direction, at a comparable frequency. A bus stop
across the road that only serves one local line is not a real substitute for a regional hub.
The test therefore **overstates** substitutability, which means the isolated count reported
here is a **lower bound** on the number of true single points of failure.

## 1. Environment bootstrap

The cell below makes the notebook runnable both on a local checkout of the repository and on
Google Colab. It locates the repository root by walking up from the current working directory
looking for the GTFS folder `israel-public-transportation`; if that fails (i.e. we are on a
fresh Colab VM) it clones the repository. It also defines `_ensure`, a helper that pip-installs
only the packages that are genuinely missing, so re-running the notebook costs nothing.

`OUT` is the root folder for all notebook-produced artifacts. The pre-existing
`outputs/tables`, `outputs/figures` and `outputs/rail` folders hold the results cited in the
written report and are never touched by these notebooks.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. Libraries

This stage is deliberately light: no graph library is needed because the graph was already
reduced to a per-stop metrics table in notebook 04. We need `pandas` for the table,
`numpy` for the radian conversion, `matplotlib` for the figures, and
`sklearn.neighbors.BallTree` for the nearest-neighbour search.

**Why a BallTree and not a brute-force distance matrix?** There are roughly 30k stops in the
graph. A full pairwise matrix would be ~30,000 x 30,000 = 9 x 10^8 distances - several
gigabytes and slow. A ball tree with the `haversine` metric answers the "nearest other stop"
query for every stop in O(n log n) and runs in a couple of seconds.

In [ ]:
_ensure("numpy", "pandas", "matplotlib", "scikit-learn")

import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

print("numpy", np.__version__, "| pandas", pd.__version__)

## 3. Hebrew text rendering in matplotlib

The GTFS stop names are in Hebrew, and one of the figures below labels individual stops by
name. Matplotlib does not implement the Unicode bidirectional algorithm, so Hebrew strings are
drawn left-to-right and come out reversed. The patch below wraps `matplotlib.text.Text.set_text`
so that any string containing Hebrew characters is converted to display order once, before it
is rendered. It is idempotent - re-running the cell will not double-apply the patch (and
double-applying would reverse the text back again).

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. Stage folders and analysis parameters

Every notebook writes into its own stage folder so that stages never overwrite each other and
a grader can see exactly which artifact came from which step.

The two parameters that define the whole analysis are collected here so they are easy to
change and easy to challenge:

| Constant | Value | Meaning |
| --- | --- | --- |
| `CRITICAL_QUANTILE` | `0.90` | "Critical" = betweenness in the top 10% of all stops. This is the project's working definition, kept consistent with the centrality stage. |
| `WALK_M` | `300` | A reasonable walking distance to an alternative stop. 300 m is roughly a 4-minute walk and is a common threshold in transit-accessibility literature. |
| `EARTH_M` | `6371000` | Mean Earth radius in metres - converts the haversine tree's radian output to metres. |
| `TOP_N_NAMED` | `15` | How many individual isolated stops to name in the second figure. |

**Cost:** everything in this notebook is cheap - the ball tree query over ~30k points takes a
few seconds and a few tens of MB. Nothing here needs sampling or approximation.

In [ ]:
STAGE = OUT / "05_critical_station_isolation"
TABLES = STAGE / "tables"
FIGURES = STAGE / "figures"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

CRITICAL_QUANTILE = 0.90      # critical = top 10% betweenness
WALK_M = 300.0                # walking distance to an acceptable alternative stop (metres)
EARTH_M = 6371000.0           # mean Earth radius (metres)
TOP_N_NAMED = 15              # how many isolated stops to name in the second figure

print("Stage folder:", STAGE)

## 5. Load the centrality metrics from stage 04

This notebook consumes `stop_metrics.csv` from the centrality stage rather than recomputing
betweenness, which is by far the most expensive quantity in the whole project. If the file is
missing we fail loudly with an actionable message instead of silently producing an empty plot.

The loader also normalises a few column-name variants (`stop_lat`/`lat`,
`approx_betweenness`/`betweenness`) so it works with either naming convention, and drops rows
without coordinates - a stop with no `lat`/`lon` cannot participate in a spatial search.
Those dropped rows are reported so the loss is visible rather than hidden.

In [ ]:
CENTRALITY_STAGE = OUT / "04_centrality_analysis"
metrics_path = CENTRALITY_STAGE / "tables" / "stop_metrics.csv"

if not metrics_path.exists():
    # tolerate a slightly different stage folder name for the centrality notebook
    candidates = sorted(OUT.glob("*centrality*/tables/stop_metrics.csv"))
    if candidates:
        metrics_path = candidates[0]
    else:
        raise FileNotFoundError(
            f"{metrics_path} missing - run notebook 04 (centrality analysis) first.")

raw = pd.read_csv(metrics_path, encoding="utf-8-sig")

# Normalise column names so either naming convention works.
RENAME = {
    "stop_lat": "lat",
    "stop_lon": "lon",
    "approx_betweenness": "betweenness",
    "betweenness_centrality": "betweenness",
}
raw = raw.rename(columns={k: v for k, v in RENAME.items()
                          if k in raw.columns and v not in raw.columns})

required = {"stop_id", "lat", "lon", "betweenness"}
missing_cols = required - set(raw.columns)
if missing_cols:
    raise KeyError(
        f"{metrics_path} is missing column(s) {sorted(missing_cols)}. "
        f"Found: {sorted(raw.columns)}")

if "stop_name" not in raw.columns:
    raw["stop_name"] = ""

metrics = raw.dropna(subset=["lat", "lon", "betweenness"]).reset_index(drop=True)

print(f"Loaded  : {metrics_path}")
print(f"Stops   : {len(raw):,} rows -> {len(metrics):,} usable "
      f"({len(raw) - len(metrics):,} dropped for missing coordinates/betweenness)")
metrics[["stop_id", "stop_name", "lat", "lon", "betweenness"]].head()

## 6. Distance to the nearest alternative stop (haversine BallTree)

For every stop we want the great-circle distance to the *closest other* stop.

1. Latitude/longitude are converted to **radians** - `sklearn`'s `haversine` metric expects
   radians and returns an angular distance on the unit sphere.
2. A `BallTree` is built over those points. A ball tree recursively partitions the points into
   nested hyperspheres, which lets a nearest-neighbour query prune whole branches instead of
   scanning every point.
3. We query with `k=2`. The nearest neighbour of a point *is the point itself* (distance 0), so
   column `0` is discarded and column `1` is the genuine nearest **other** stop.
4. Multiplying the returned angular distance by the Earth radius converts it to metres.

Note that two distinct `stop_id`s can share identical coordinates (opposite sides of a road are
sometimes coded to the same point), which yields a distance of 0 m - a legitimate "there is an
alternative right here" result under this spatial test.

In [ ]:
coords = np.radians(metrics[["lat", "lon"]].to_numpy(dtype=float))

tree = BallTree(coords, metric="haversine")
dist, _ = tree.query(coords, k=2)          # k=2: the stop itself + its nearest neighbour
metrics["nearest_alt_m"] = dist[:, 1] * EARTH_M

print("Distance to nearest other stop (metres), across all graph stops:")
print(metrics["nearest_alt_m"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).round(1))

## 7. Split the critical stops into substitutable vs isolated

"Critical" is defined as **betweenness at or above the 90th percentile** - the top 10% of stops
by how many shortest paths run through them. Among those, a stop is **isolated** when its
nearest alternative is farther than `WALK_M` (300 m): losing it means the surrounding demand
has nowhere convenient to go.

One small consistency fix relative to the original script: the histogram bins below are
closed on the right (`right=True`), so a stop at exactly 300.0 m falls in the green
`100-300 m` band, matching the `> 300 m` isolation test. In the original script the bins were
closed on the left, so a hypothetical stop at exactly 300.0 m would have been drawn red while
being counted as non-isolated. It affects at most a measure-zero set of stops but the numbers
in the table and the figure now agree by construction.

In [ ]:
threshold = metrics["betweenness"].quantile(CRITICAL_QUANTILE)
critical = metrics[metrics["betweenness"] >= threshold].copy()
critical["is_isolated"] = critical["nearest_alt_m"] > WALK_M

n_crit = len(critical)
n_isolated = int(critical["is_isolated"].sum())
n_covered = n_crit - n_isolated

print(f"Betweenness threshold (q={CRITICAL_QUANTILE:.2f}) : {threshold:.8f}")
print(f"Critical stops (top 10% betweenness)   : {n_crit:,}")
print(f"  with an alternative within {WALK_M:.0f} m    : {n_covered:,} "
      f"({100 * n_covered / n_crit:.1f}%)")
print(f"  isolated (true single point of fail.) : {n_isolated:,} "
      f"({100 * n_isolated / n_crit:.1f}%)")

## 8. Save the tables

Two artifacts are written:

- **`critical_isolation.csv`** - the full list of critical stops sorted by betweenness, with
  `nearest_alt_m` and `is_isolated`. This is the row-level evidence behind every claim made
  from this stage, and it is the table that should be consulted before naming any specific
  station as a weak point.
- **`isolation_summary.csv`** - the five distance bands with their counts and shares; this is
  exactly what the main figure plots, saved separately so the figure can be checked against
  numbers rather than pixels.

Both are written with `utf-8-sig` so that Hebrew stop names open correctly in Excel.

In [ ]:
BINS = [0, 100, 300, 500, 1000, np.inf]
BAND_LABELS = ["< 100 m", "100-300 m", "300-500 m", "500-1000 m", "> 1 km"]

critical["distance_band"] = pd.cut(critical["nearest_alt_m"], bins=BINS,
                                   labels=BAND_LABELS, right=True, include_lowest=True)

critical_sorted = critical.sort_values("betweenness", ascending=False)
critical_path = TABLES / "critical_isolation.csv"
critical_sorted.to_csv(critical_path, index=False, encoding="utf-8-sig")

band_counts = critical["distance_band"].value_counts().reindex(BAND_LABELS).fillna(0).astype(int)
summary = pd.DataFrame({
    "distance_band": BAND_LABELS,
    "n_critical_stops": band_counts.to_numpy(),
    "share_pct": (100 * band_counts.to_numpy() / n_crit).round(2),
    "verdict": ["substitutable", "substitutable", "isolated", "isolated", "isolated"],
})
summary_path = TABLES / "isolation_summary.csv"
summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("wrote", critical_path)
print("wrote", summary_path)
summary

## 9. Main figure - does the critical stop have a walkable alternative?

This is the figure used in the final presentation. Each bar is a band of "distance from a
critical stop to its nearest alternative stop", and the bar height is how many critical stops
fall in that band. The colour encodes the verdict rather than the value: **green** for the two
bands inside the 300 m walking threshold (the stop is, spatially at least, replaceable) and
**red** for the three bands beyond it (no alternative in walking distance).

Counts and percentages are printed on top of the bars because the point of the chart is the
*ratio* between green and red, not the absolute heights.

In [ ]:
counts = band_counts.to_numpy()
colors = ["#16a34a", "#16a34a", "#dc2626", "#dc2626", "#dc2626"]

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.bar(BAND_LABELS, counts, color=colors, edgecolor="white")

offset = max(counts.max() * 0.02, 1)
for bar, v in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + offset,
            f"{v:,}\n({100 * v / n_crit:.0f}%)", ha="center", va="bottom", fontsize=11)

ax.set_xlabel("Distance to the nearest alternative stop", fontsize=14)
ax.set_ylabel("Number of critical stops", fontsize=14)
ax.set_title("Does a critical stop have an alternative within walking distance?",
             fontsize=16, fontweight="bold", pad=10)
ax.set_ylim(0, counts.max() * 1.18)
ax.tick_params(labelsize=12)
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="#16a34a", label=f"Alternative within {WALK_M:.0f} m - substitutable in practice"),
    Patch(color="#dc2626", label="Isolated - genuine single point of failure"),
], fontsize=12, loc="upper right")

plt.tight_layout()
fig_path = FIGURES / "critical_isolation.png"
plt.savefig(fig_path, dpi=150)
plt.show()
print("wrote", fig_path)

## 10. Which isolated stops matter most?

The aggregate chart says *how many* critical stops are isolated; it does not say *which*. This
figure names the isolated critical stops with the highest betweenness - the stops where high
structural load and the absence of a walkable alternative coincide. Those are the concrete
candidates for redundancy investment (an added parallel service, a second stop, a shuttle).

Bars are betweenness (structural importance); the annotation on each bar is the distance to the
nearest alternative stop, so both halves of the criterion are visible at once. Stop names are
Hebrew and are rendered through the bidi patch installed in section 3; where a name is missing
we fall back to the `stop_id`.

In [ ]:
isolated = critical_sorted[critical_sorted["is_isolated"]].copy()

if isolated.empty:
    print("No isolated critical stops found - nothing to plot.")
else:
    top = isolated.head(TOP_N_NAMED).copy()
    top["label"] = (top["stop_name"].fillna("").astype(str).str.strip()
                    .where(lambda s: s != "", top["stop_id"].astype(str)))
    top = top.sort_values("betweenness")   # smallest at the bottom of a barh

    fig, ax = plt.subplots(figsize=(11, 7))
    ax.barh(top["label"], top["betweenness"], color="#dc2626")

    span = top["betweenness"].max()
    for y, (btw, d) in enumerate(zip(top["betweenness"], top["nearest_alt_m"])):
        ax.text(btw + span * 0.01, y, f"{d:,.0f} m to nearest alt.",
                va="center", fontsize=10, color="#374151")

    ax.set_xlim(0, span * 1.28)
    ax.set_xlabel("Betweenness centrality", fontsize=13)
    ax.set_title(f"Top {len(top)} isolated critical stops "
                 f"(top 10% betweenness, no alternative within {WALK_M:.0f} m)",
                 fontsize=15, fontweight="bold", pad=10)
    ax.grid(axis="x", alpha=0.3)
    ax.set_axisbelow(True)

    plt.tight_layout()
    fig2_path = FIGURES / "top_isolated_critical_stops.png"
    plt.savefig(fig2_path, dpi=150)
    plt.show()
    print("wrote", fig2_path)

    display_cols = [c for c in ["stop_id", "stop_name", "betweenness", "nearest_alt_m"]
                    if c in isolated.columns]
    isolated[display_cols].head(TOP_N_NAMED).round(6)

## 11. Limitations - read this before quoting any number above

**1. The substitutability test is purely spatial.** "There is another stop within 300 m" says
nothing about whether that stop serves the *same lines*, in the *same direction*, at a
*comparable frequency*, or at the same hours. A regional interchange and a single-line local
stop 120 m away are treated as interchangeable here, and they plainly are not. The test
therefore **systematically overstates** how replaceable critical stops are: the green bars are
too tall and the red bars are too short. The isolated count should be read as a **lower bound**
on the number of genuine single points of failure. A stricter version would require the
neighbouring stop to share at least one `route_id` with the critical stop.

**2. The BallTree is built only over stops present in the graph**, not over all GTFS stops.
Stops that were filtered out during graph construction (isolated stops, stops with no usable
trips, deduplicated entries) are invisible to the nearest-neighbour search. If such a stop sits
next door to a critical stop, that critical stop is scored as isolated when in reality a
physical stop exists nearby. This biases in the opposite direction to limitation 1 and its
magnitude is not quantified here.

**3. Straight-line distance, not walking distance.** Haversine measures as the crow flies. A
stop 250 m away across a motorway, a rail line or a river is not a 250 m walk. Real network
walking distance would only ever be longer, so this again over-counts substitutability.

**4. Betweenness is an approximation.** The upstream centrality stage samples source nodes
rather than computing exact betweenness, so membership of the top 10% is stable for the clearly
dominant hubs but noisy near the 90th-percentile cut-off. Stops sitting just either side of the
threshold should not be treated as definitively in or out.

**5. The 300 m and top-10% cut-offs are conventions, not findings.** They are exposed as
constants in section 4 precisely so the reader can re-run with 500 m or a top-5% definition and
see how much the headline ratio moves.

## 12. Takeaways

- **Graph criticality and real-world criticality are not the same thing.** Under a purely
  spatial substitutability test, a large share of the stops the graph flags as critical have
  another stop within a short walk. For those, the "remove the node and the network breaks"
  story from centrality analysis overstates the operational consequence: passengers walk 2-4
  minutes and carry on. Run the notebook to read the exact split off section 7 - the point of
  the figure is the green/red ratio, and it is the *ratio* that should be quoted, not a number
  remembered from a previous run.
- **The residual red group is the real finding.** The critical stops with no alternative inside
  300 m are a much smaller, much more actionable list than "the top 10% by betweenness", and
  `tables/critical_isolation.csv` names them. This is the set worth targeting with redundancy
  measures.
- **Be honest about the direction of the error.** The spatial test is generous - it counts any
  nearby stop as a substitute regardless of which lines it serves - so the isolated group is a
  floor, not a ceiling. The counter-bias (only graph stops are in the tree) pushes the other
  way but is smaller and unquantified. The correct summary is: *this analysis narrows the
  candidate list of true single points of failure; it does not finalise it.*
- **Method note.** The interesting part of this stage is cheap: a haversine ball tree turns a
  would-be 9 x 10^8-pair distance problem into a few seconds of work, which is what makes it
  practical to combine a structural metric with a geographic one at all.